# Basics of Laue Pattern peak search and Unit cell Refinement

## This Notebook is a part of Tutorials on LaueTools Suite.  
Author: J.-S. Micha

Last Revision:   August 2019

tested with python3

**Objectives**

- Load and display Laue pattern images
- Perform a Peak Search
- Perform the indexation of a Laue spots list
- Perform the crystal orientation and unit cell refinement 

Setting absolute path to LaueTools Modules if Lauetools has not been installed with pip. It is assumed that this notebook is located in a subfolder (normally Notebooks)

In [ ]:
LaueToolsCode_Folder = '..'
import sys,os
abspathLaueTools =os.path.abspath(LaueToolsCode_Folder)
print('abspathLaueTools',abspathLaueTools)
sys.path.append(LaueToolsCode_Folder)

In [ ]:
import LaueTools
LaueTools.__file__

In [ ]:
#%matplotlib inline
%matplotlib notebook

import time,copy,os

# Third party modules
import matplotlib     # graphs and plots
import matplotlib.pyplot as plt
import numpy as np    # numerical arrays

# LaueTools modules

import LaueTools.IOLaueTools as IOLT   # read and write ASCII file  (IO) 
import LaueTools.readmccd as RMCCD # read CCD and detector binary file, PeakSearch methods


Considering single image analysis (that belong to the LaueTools distribution)

In [ ]:
t0 = time.time()
LaueToolsExamplesFolder = os.path.join(LaueToolsCode_Folder,'Examples')

imageindex = None
imagefolder =os.path.join(LaueToolsCode_Folder,'LaueImages')
imagefilename = 'Ge_blanc_0000.mccd'

#imagefolder =os.path.join(LaueToolsCode_Folder,'LaueImages')
#imagefilename = 'CdTe_I999_03Jul06_0200.mccd'

Considering analysis of one image in dataset

**For information:** select image file of interest, in case of set of images with index. Then, splitting imagefilename allows to loop over images:  prefix+index.extension

In [ ]:
%%script false
# just to show (cell not executed)

imagefolder ='/home/micha/LaueProjects/VO2/ToScript/Data_VO2'

prefixfilename= 'CT30_'
imageindex=20

imagefilename = prefixfilename+'%04d.mccd'%imageindex
print("imagefilename :",imagefilename)
# you should see: imagefilename : CT30_0020.mccd


**Read image file, get data and display it**

Function `readCCDimage()` returns `dataimage` as a 2D numpy array with the proper dimensions and orientation given by `framedim` and the geometrical transformations labelled by `fliprot`

In [ ]:
print('Displaying %s\n'%imagefilename)
dataimage, framedim, fliprot = RMCCD.readCCDimage(imagefilename,dirname=imagefolder,CCDLabel='MARCCD165')
fullpathimagefile= os.path.join(imagefolder,imagefilename)

fig, ax = plt.subplots(figsize=(4,4))

ax.imshow(dataimage,vmin=0,vmax=2000)
ax.set_title('%s'%imagefilename)

***peaksearch*** is performed in two main steps:
- 1) blobs or local maxima finder
- 2) for blob, refinement starting from blob average center.

For the first step, `readCCDimage()` is called to obtain raw data if no different data array is provided with the argument `Data_for_localMaxima` (set to `None` by default). After second step, Peaksearch results can be purged from peaks already present in a file as an optional argument `Remove_BlackListedPeaks_fromfile`.

In [ ]:
import os
ti1= time.time()

#blacklistedpeaksfile=os.path.join(folder,'Blacklist.dat')
blacklistedpeaksfile = None

res=RMCCD.PeakSearch(fullpathimagefile,CCDLabel='MARCCD165',
                     return_histo=0,local_maxima_search_method=0,
                     IntensityThreshold=200,
                     boxsize=5,
                     fit_peaks_gaussian=1,
                     FitPixelDev=10,
                     Data_for_localMaxima=None,#newdataimage,
                     Remove_BlackListedPeaks_fromfile=blacklistedpeaksfile)
tps =time.time()
print("peak search time",tps-ti1)

**Spots properties**:

peak_X, peak_Y, peak_I, peak_fwaxmaj, peak_fwaxmin, peak_inclination, Xdev, Ydev, peak_bkg, Ipixmax,

Spots are sorted by intensity (according to the 2D gaussian fit)

In [ ]:
peaklist=res[0]
print('Digital Spots properties for the 5 most intense spots')
print(peaklist[:6])

In [ ]:
print('X, Y pixel refinement positions for the first 5 spots')
peaklist[:5,:2]

*add markers to image*

In [ ]:
if len(peaklist)<=1: raise ValueError

#datatoplot=newdataimage
datatoplot = dataimage
    
fig, ax = plt.subplots()
ax.imshow(datatoplot,vmin=0,vmax=1000,cmap='hot')

from matplotlib.patches import Circle

F=plt.gcf()
axes=F.gca()
F.get_dpi()
defaultSize=F.get_size_inches()
F.set_size_inches(defaultSize*1.5)

# delete previous patches:

axes.patches = []

# rebuild circular markers
largehollowcircles = []
smallredcircles = []
# correction only to fit peak position to the display
offset_convention = np.array([1, 1])

XYlist = peaklist[:, :2] - offset_convention

for po in XYlist:

    large_circle = Circle(po, 7, fill=False, color='b')
    center_circle = Circle(po, .5 , fill=True, color='r')
    axes.add_patch(large_circle)
    axes.add_patch(center_circle)

    largehollowcircles.append(large_circle)
    smallredcircles.append(center_circle)

**List of peaks props is written in a file with extension .dat, here the variable is `datfilename`**

In [ ]:
if imageindex is not None:
    peaklistprefix=prefixfilename+'cor_%04d'%imageindex
else:
    peaklistprefix=imagefilename.split('.')[0]+'Notebook'
print('peaklist.shape',peaklist.shape)
print("fullpathimagefile",fullpathimagefile)
print('imagefolder',imagefolder)
RMCCD.writepeaklist(peaklist,peaklistprefix,outputfolder=imagefolder,initialfilename=fullpathimagefile)

datfilename = peaklistprefix+'.dat'

### Now indexing

##### geometry calibration parameters

Either you fill manually the dict of parameters or you read a file  .det

In [ ]:
# detector geometry and parameters as read from Geblanc0000.det
calibration_parameters = [70.775, 941.74, 1082.57, 0.631, -0.681]
CCDCalibdict = {}
CCDCalibdict['CCDCalibParameters'] = calibration_parameters
CCDCalibdict['framedim'] = (2048, 2048)
CCDCalibdict['detectordiameter'] = 165.
CCDCalibdict['kf_direction'] = 'Z>0'
CCDCalibdict['xpixelsize'] = 0.07914

# CCDCalibdict can also be simply build by reading the proper .det file
print("reading geometry calibration file")
CCDCalibdict=IOLT.readCalib_det_file(os.path.join(imagefolder,'Geblanc0000.det'))
CCDCalibdict['kf_direction'] = 'Z>0'

**creation of a .cor file containing accurate scattering angles thanks to detector geometry parameters**

Only list of spots with scattering angles can be indexed. In LaueTools .dat file contains only X, Y pixel positions, .cor file contains in addition 2theta and chi scattering angles, and .fit file in addition indexed results properties (such as h, k, l, energy, grain index ...)

In [ ]:
import LaueTools.LaueGeometry as LTGeo
LTGeo.convert2corfile(datfilename,
                         calibration_parameters,
                         dirname_in=imagefolder,
                        dirname_out=imagefolder,
                        CCDCalibdict=CCDCalibdict)
corfilename = datfilename.split('.')[0] + '.cor'
fullpathcorfile = os.path.join(imagefolder,corfilename)

#### create instance of an objet spotsset class

In [ ]:
import LaueTools.indexingSpotsSet as ISS
DataSet = ISS.spotsset()

DataSet.importdatafromfile(fullpathcorfile)

In [ ]:
DataSet.getUnIndexedSpotsallData()[:3]

***Set parameters for indexation: Ge, maximum energy***

All materials are listed in dict_LaueTools.py in dict_Materials. You can edit/modify the module (then a restart of the kernel is necessary)

In [ ]:
emin=5
# emax can be lowered for large unit cell indexation (but greater than BM32 highest energy is meaningless)
emax=22
# key of materials 
key_material='Ge'

dict_indexrefine = {# recognition angle parameters from two sets A and B
                   'AngleTolLUT': 0.5,
                   'nlutmax':3,
                   'central spots indices': [0,1,2,3,4],  # spots set A 
                   'NBMAXPROBED': 10,  # spots set B
                   'MATCHINGRATE_ANGLE_TOL': 0.2,
                # refinement parameters (loop over narrower matching angles)
                   'list matching tol angles':[0.5,0.2,0.1],
               
                # minor parameters
                'MATCHINGRATE_THRESHOLD_IAL': 100,
                   'UseIntensityWeights': False,
                   'nbSpotsToIndex':10000,
                   'MinimumNumberMatches': 3,
                   'MinimumMatchingRate':3
                   }

#
grainindex=0
DataSet = ISS.spotsset()
    
DataSet.pixelsize = CCDCalibdict['xpixelsize']
DataSet.dim = CCDCalibdict['framedim']
DataSet.detectordiameter = CCDCalibdict['detectordiameter']
DataSet.kf_direction = CCDCalibdict['kf_direction']
DataSet.key_material = key_material
DataSet.emin = emin
DataSet.emax = emax

**Before launching the indexation procedure you may want to check a solution found elsewhere or sometimes ago. Then fill `previousResults` as shown below**

In [ ]:
#CheckFirstThisMatrix=np.array([[-0.44486058225058 ,  0.098996190230096 ,-0.897868909077371],[-0.883970521873963,0.1130536332378 , 0.462465547362675],
# [ 0.143878606007886, 0.993706753289519 , 0.035064809225047]])

# nb of matrices, list of matrices to check, dummy parameter, dummy parameter
#previousResults = 1,[CheckFirstThisMatrix],50,50


previousResults = None

**Then launch indexation by specifying some arguments of the method `IndexSpotsSet`:**

    - nbGrainstoFind: nb of grains of this material you want to find
    - set_central_spots_hkl: imposed miller indices [h,k,l] of central spots (set A of spots)  else : None
    ...

In [ ]:
t0 =time.time()

DataSet.IndexSpotsSet(fullpathcorfile, key_material, emin, emax, dict_indexrefine, None,
                         use_file=1, # read .cor file and reset also spots properties dictionary
                         IMM=False,LUT=None,n_LUT=dict_indexrefine['nlutmax'],angletol_list=dict_indexrefine['list matching tol angles'],
                        nbGrainstoFind=1,  # nb of grains of the same material in this case
                        set_central_spots_hkl=[0,1,1],  # set hkl of spots of set A
                        MatchingRate_List=[10, 10, 10,10,10,10,10,10],  # minimum matching rate figure to keep on looping for refinement
                        verbose=0,
                        previousResults=previousResults, # check before the orientation if not None
                        corfilename=corfilename)

# write unindexed spots list in a .cor file
DataSet.writecorFile_unindexedSpots(corfilename=corfilename,
                                                dirname=imagefolder,
                                                filename_nbdigits=4)

# write .fit file of indexed spots belonging to grain #0
DataSet.writeFitFile(0,corfilename=corfilename,dirname=imagefolder)

tf = time.time()-t0

In [ ]:
print('Indexation time %.3f second(s) \n\n'%tf)
print('Spots properties of the 10 first spots that have been indexed (sorted by intensity)')
print('#spot 2theta chi X, Y intensity h k l energy')
print(DataSet.getSpotsFamilyallData(0)[:10])

DataSet is an object with many attributes and methods related to spots properties (indexed or not, belonging to grains counted from zero). By press Tab key after having typed DataSet. can show you infos about spots

In [ ]:
DataSet.B0matrix